# 정답 픽셀 없는 획 구조 유도 — 일반화 시험

표준 글자의 획 추출 결과에서 정규화한 방향·비대칭·상대 위치·접촉 위치를 구합니다.
생성물에서 같은 획 추출 모델로 구한 특성의 smooth L1 손실을 최적화합니다.
표준 글자 RGB와의 픽셀 손실·삭제 지도·정답 윤곽 맞추기는 보정에 사용하지 않습니다.
이 지표는 기하 모멘트 기반 대리 지표이며 완전한 위상/문자 인식 지표가 아닙니다.

기본 INIT_MODE='noise': 표준 이미지를 VAE로 인코딩하지 않고 잡음에서 생성합니다.
3번 셀에서 선택한 네 표준 글자와 동일한 문자를 순서대로 포함한 영문 프롬프트를 입력하세요.
INIT_MODE='external'이면 같은 2×2 배치의 다른 폰트 이미지 경로를 INPUT_SHEET에 넣습니다.
외부 이미지의 정답 획/깨끗한 짝 이미지는 필요하지 않습니다. 문자 순서·배치는 사용자가 맞춰야 합니다.

기존 훈련 목록의 표준 글자를 조건으로 사용하므로 미학습 문자 일반화가 입증되는 실험은 아닙니다.
다른 폰트/잡음 seed의 성공률, 정상 획 손상, 문자 판독 결과로 평가해야 합니다.
SD1.5가 네 글자를 만들지 못한 결과도 제외하지 않습니다. 템플릿으로 형태가 끌리는 편향이 남을 수 있습니다.
실제 GPU 품질은 아직 검증 전입니다. 1→5번 셀을 실행하세요.

In [ ]:
from pathlib import Path
import subprocess, sys, tempfile, json
root = Path(tempfile.mkdtemp(prefix='hsqm_diffusion_score_', dir='/content'))
subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'main',
    'https://github.com/yechan25/hsqm.git', str(root)], check=True)
commit = subprocess.check_output(['git', '-C', str(root), 'rev-parse', 'HEAD'], text=True).strip()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'diffusers==0.35.1', 'transformers==4.49.0', 'accelerate', 'safetensors',
    'timm', 'pandas', 'numpy', 'tqdm', 'matplotlib', 'pillow', 'scipy',
    'scikit-image', 'pyyaml', 'fonttools', 'ipywidgets'], check=True)
sys.path.insert(0, str(root))
for key in list(sys.modules):
    if key.split('.')[0] in ('stroke_model', 'font_test', 'diffusion_model'):
        del sys.modules[key]
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Colab에서 GPU 런타임으로 바꿔주세요.')
print('GPU:', torch.cuda.get_device_name(0))
print('실행 GitHub 커밋:', commit)
from google.colab import drive, output
drive.mount('/content/drive')
output.enable_custom_widget_manager()

In [ ]:
from stroke_model.convert_reference_csv import existing_path
from diffusion_model.score_pilot import training_catalog, generate_pilot, score_pilot, load_catalog_reference
from diffusion_model.pilot_data import prepare_pilot_data
from diffusion_model.multiglyph_pilot import generate_multiglyph_pilot
from IPython.display import display
import ipywidgets as widgets
HSQM = existing_path('/content/drive/Shareddrives/2026 자율연구/HSQM')
CHECKPOINT = None  # 폰트 실험에서 사용한 best.pt의 전체 경로를 넣으면 그 모델 사용
DATA_ROOT = HSQM/'dataset/automated_train_dataset'  # 기존 학습 ZIP/TAR가 있는 폴더
CSV_PATH = None  # None: 체크포인트 옆에 저장된 학습 CSV를 자동 사용
TRAIN_MANIFEST = None  # 기존 세션의 유효한 목록이 있을 때만 선택적으로 지정. 없어도 자동 재생성
REFERENCE_ONLY = True  # 시험에는 기준 글자/기준 획만 사용. 전체 학습 NPZ 변환 생략
SCORE_CONFIG = {}  # 폰트 시험에서 값을 바꿨다면 동일한 설정 입력
SOURCE_CONFIG = {}
GLYPH_COUNTS = [4]
CANVAS_SIZE = 576
FIXED_CELL_PIXELS = None  # None: 전체 크기 고정. 192: 글자당 크기 고정 비교
STRENGTHS = [.65]
SEEDS = [100]
STEPS = 50
MODEL_ID = 'stable-diffusion-v1-5/stable-diffusion-v1-5'
if CHECKPOINT is None:
    candidates = sorted((HSQM/'v2_topology_runs').glob('*/best.pt'))
    if not candidates:
        raise FileNotFoundError('CHECKPOINT에 기존 Topo best.pt를 지정하세요.')
    CHECKPOINT = candidates[-1]
CHECKPOINT = existing_path(CHECKPOINT)
print('사용할 획 모델:', CHECKPOINT, flush=True)
preparation = prepare_pilot_data(CHECKPOINT, DATA_ROOT, csv_path=CSV_PATH, manifest=TRAIN_MANIFEST,
    reference_only=REFERENCE_ONLY)
TRAIN_MANIFEST = Path(preparation['manifest'])
print('데이터 준비 완료. 3번 셀에서 기준 글자를 선택하세요.', flush=True)

# Separate experimental path; old pixel-control notebook is preserved.
SEED=100  # Repeat with 101, 102, ...; keep all results
INIT_MODE='noise'  # 'external' for a different font/input sheet
INPUT_SHEET=None
GUIDANCE=dict(local_edit=False,clean_step_ratio=.01,start_fraction=.5,end_fraction=1.,
              every=1,backtracks=5,state_update='clean_residual',max_new_local_ink=None)

In [ ]:
import io, math
import numpy as np
from PIL import Image, ImageDraw
catalog = training_catalog(TRAIN_MANIFEST)
by_row = {item['manifest_row']: item for item in catalog}
needed = max(GLYPH_COUNTS)
if len(catalog) < needed:
    raise ValueError(f'사용 가능한 학습 기준은 {len(catalog)}개입니다. GLYPH_COUNTS를 줄이세요.')
reference_select = widgets.SelectMultiple(
    options=[(f"train row {item['manifest_row']} | {item['char_id'] or '(그림 확인)'}", item['manifest_row']) for item in catalog],
    value=tuple(item['manifest_row'] for item in catalog[:needed]), rows=12,
    description='여러 글자:', layout=widgets.Layout(width='98%'))
preview = widgets.Image(format='png', width=500, height=500)
def update_preview(change=None):
    selected = list(reference_select.value)[:needed]
    side = math.ceil(math.sqrt(needed))
    sheet = Image.new('RGB', (160*side, 180*side), 'white')
    draw = ImageDraw.Draw(sheet)
    for index, row in enumerate(selected):
        reference,_ = load_catalog_reference(by_row[row])
        glyph = Image.fromarray(np.uint8((1-reference[0])*255)).convert('RGB')
        x, y = (index%side)*160, (index//side)*180
        sheet.paste(glyph.resize((160,160)), (x,y))
        draw.text((x+4,y+160), f'slot {index+1} | row {row}', fill='black')
    buffer=io.BytesIO(); sheet.save(buffer,format='PNG'); preview.value=buffer.getvalue()
reference_select.observe(update_preview,names='value'); update_preview()
display(widgets.VBox([reference_select,preview]))
print('정확히 4개를 선택하세요. 같은 4개로 기본/유도 경로를 비교합니다.')
print('문자명이 없는 경우 그림으로 확인하세요. 서로 다른 글자를 비교하려면 같은 글자의 폰트 변형을 중복 선택하지 마세요.')
import ipywidgets as widgets
prompt_box=widgets.Textarea(description='Prompt',placeholder='Describe the SAME four selected characters in 2x2 reading order',layout=widgets.Layout(width='95%',height='90px'))
display(prompt_box)

In [ ]:
from datetime import datetime
from huggingface_hub import model_info
from diffusion_model.run_topo_guidance import generate_guided_pair
if not prompt_box.value.strip():raise ValueError('선택한 4문자와 순서가 일치하는 프롬프트를 입력하세요.')
OUTPUT_DIR=HSQM/'structure_generalization'/datetime.now().strftime('%Y%m%d_%H%M%S_%f')
generation=generate_guided_pair(TRAIN_MANIFEST,list(reference_select.value),CHECKPOINT,OUTPUT_DIR,
    model_id=MODEL_ID,revision=model_info(MODEL_ID).sha,steps=STEPS,canvas_size=CANVAS_SIZE,
    guidance=GUIDANCE,seed=SEED,guidance_signal='template_structure',init_mode=INIT_MODE,
    input_sheet=INPUT_SHEET,prompt_text=prompt_box.value.strip(),
    appearance_weight=0,extra_weight=0,source_spill_weight=0,direct_reference=False,
    compare_structure=False,compare_uniform=False)
(OUTPUT_DIR/'git_commit.txt').write_text(commit,encoding='utf-8')
import matplotlib.pyplot as plt
fig,axes=plt.subplots(1,len(generation['pages']),figsize=(15,5))
for ax,page in zip(axes,generation['pages']):
    with Image.open(page['input']) as im:ax.imshow(im.convert('RGB'))
    ax.set_title('Template (conditioning only)' if page['method']=='reference' else page['method']);ax.axis('off')
plt.show()
print('Primary audit:',generation['structure_audit'])
print('Template encoded as input:',generation['reference_encoded_as_input'])
print('Pixel audit is auxiliary only; it is NOT the optimization metric.')

In [ ]:
import json,zipfile,pandas as pd
from google.colab import files
rows=[]
for method,data in generation['structure_audit'].items():
    for slot,loss in enumerate(data['per_glyph']):rows.append(dict(method=method,slot=slot+1,descriptor_loss=loss))
frame=pd.DataFrame(rows);display(frame)
frame.to_csv(OUTPUT_DIR/'descriptor_scores.csv',index=False)
archive=OUTPUT_DIR.parent/(OUTPUT_DIR.name+'_structure_debug.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for p in OUTPUT_DIR.rglob('*'):
        if p.is_file():z.write(p,p.relative_to(OUTPUT_DIR))
files.download(str(archive))